In [ ]:
!pip install bs4

In [ ]:
from google.colab import drive
import os
import json
drive.mount('/content/drive')

def load_api_keys():
    key_path = '/content/drive/MyDrive/api/api_keys.json'
    if os.path.exists(key_path):
        with open(key_path, 'r') as f:
            return json.load(f)
    else:
        print("API 키 파일이 Drive에 없습니다. 샘플 키를 사용합니다.")
        return {
            "KOPIS_service_key": "sample_key_1",
            "naver_client_id": "sample_key_2",
            "naver_client_secret": "sample_key_3"
        }

api_keys = load_api_keys()
service_key = api_keys["KOPIS_service_key"]
client_id = api_keys["naver_client_id"]
client_secret = api_keys["naver_client_secret"]

Mounted at /content/drive


1.   네이버 검색 api를 활용하여 뮤지컬 기사를 모두 따옴
2.   '예매' 'OST' 등 뮤지컬에서 쓸법한 키워드에 가중치를 부여하여 '뮤지컬 홍보 기사'임을 판별하기 위한 커트라인을 만들어둠. (현재 가중치가 제대로 작동하지 않아, 제목에 뮤지컬이 있나 정도만 판별해줌. 수정필)
3. 이 단계에서 바로 기사 URL에 들어가 문단별로 저장하도록 코드를 만들었으나, 전혀 작동하지 않음.



In [ ]:
import csv
import requests
from datetime import datetime, timedelta
import json
import re
from bs4 import BeautifulSoup
import time

def remove_brackets(text): # 기능 대체시킬 예정
    return re.sub(r'\[.*?\]|\(.*?\)', '', text).strip()

def remove_html_tags(text):
    return re.sub('<[^<]+?>', '', text)

def get_full_article_content(url, timeout=10):
    try:
        response = requests.get(url, timeout=timeout)
        soup = BeautifulSoup(response.text, 'html.parser')
        article_body = soup.find('article', id='dic_area')
        if article_body:
            paragraphs = article_body.find_all('p')
            return [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)]
    except Exception as e:
        print(f"An error occurred while processing URL {url}: {str(e)}")
    return []

def search_news(query, start_date, end_date, client_id, client_secret):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }

    articles = []
    start = 1
    display = 100

    while True:
        params = {
            "query": query,
            "sort": "date",
            "start": start,
            "display": display,
            "start_date": start_date.strftime("%Y%m%d"),
            "end_date": end_date.strftime("%Y%m%d")
        }

        response = requests.get(url, headers=headers, params=params)
        result = json.loads(response.text)

        print(f"API Response for query '{query}': {result.get('total', 0)} total results")

        if "items" in result:
            articles.extend(result["items"])
            print(f"Retrieved {len(result['items'])} articles in this request")

            if len(result["items"]) < display:
                break

            start += display
        else:
            print(f"No items found in the response. Response: {result}")
            break

    return articles

def calculate_article_score(article, musical_name, theater_name):
    content = ' '.join(article['full_content']).lower()
    title = article['title'].lower()
    musical_name = musical_name.lower()
    theater_name = theater_name.lower()

    score = 0

    # 제목에 뮤지컬 이름이 있으면 높은 점수
    if musical_name in title:
        score += 10

    # 키워드 기반 점수
    keywords = {
        musical_name: 5,
        theater_name: 3,
        "뮤지컬": 4,
        "공연": 3,
        "티켓": 2,
        "예매": 2,
        "배우": 1,
        "출연": 1,
        "관람": 1,
        "무대": 1,
        "연출": 1,
    }

    for keyword, points in keywords.items():
        score += content.count(keyword) * points

    # 문맥 분석
    sentences = re.split(r'[.!?]+', content)
    for i, sentence in enumerate(sentences):
        if musical_name in sentence and "뮤지컬" in sentence:
            score += 5
        if i > 0 and musical_name in sentence and "뮤지컬" in sentences[i-1]:
            score += 3

    return score

def get_musical_news(musical_name, theater_name, start_date, end_date, client_id, client_secret):
    query = f'"{musical_name}" "{theater_name}"'
    print(f"\nSearching for: {query}")
    print(f"Date range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

    articles = search_news(query, start_date, end_date, client_id, client_secret)

    filtered_articles = []
    for i, article in enumerate(articles):
        print(f"\nChecking article {i+1}: {article['link']}")
        article['full_content'] = get_full_article_content(article['link'])

        score = calculate_article_score(article, musical_name, theater_name)
        article['score'] = score
        print(f"Article {i+1} score: {score}")

        if score >= 10:  # 임계값 설정
            filtered_articles.append(article)
            print(f"Article {i+1}: Added to filtered list")
        else:
            print(f"Article {i+1}: Filtered out")

    filtered_articles.sort(key=lambda x: x['score'], reverse=True)
    print(f"Total filtered articles: {len(filtered_articles)}")
    return filtered_articles

def save_to_csv(musicals_data, file_path):
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Musical Name', 'Theater Name', 'Article Title', 'Publication Date', 'Article URL', 'Score'] +
                        [f'Paragraph {i}' for i in range(1, 51)])  # Up to 50 paragraphs

        for musical in musicals_data:
            for article in musical['articles']:
                row = [
                    musical['name'],
                    musical['theater'],
                    remove_html_tags(article['title']),
                    article['pubDate'],
                    article['link'],
                    article['score']
                ]
                for paragraph in article['full_content'][:50]:
                    row.append(paragraph)
                writer.writerow(row)

def main():

    musicals_data = [
        {
            'name': '프랑켄슈타인',
            'theater': '블루스퀘어',
            'start_date': '2024.06.05',
            'end_date': '2024.08.25'
        },
        {
            'name': '하데스타운 [서울]',
            'theater': '샤롯데씨어터',
            'start_date': '2024.07.12',
            'end_date': '2024.10.06'
        }
    ]

    for musical in musicals_data:
        name = remove_brackets(musical['name'])
        theater = remove_brackets(musical['theater'])
        start_date = datetime.strptime(musical['start_date'], "%Y.%m.%d") - timedelta(days=90)
        end_date = datetime.strptime(musical['end_date'], "%Y.%m.%d") + timedelta(days=30)

        print(f"\nProcessing: {name} at {theater}")
        articles = get_musical_news(name, theater, start_date, end_date, client_id, client_secret)
        musical['articles'] = articles

        print(f"Processed: {name}, Found articles: {len(articles)}")

    save_to_csv(musicals_data, "musical_articles.csv")

if __name__ == "__main__":
    main()


Processing: 프랑켄슈타인 at 블루스퀘어

Searching for: "프랑켄슈타인" "블루스퀘어"
Date range: 2024-03-07 to 2024-09-24
API Response for query '"프랑켄슈타인" "블루스퀘어"': 2608 total results
Retrieved 100 articles in this request
API Response for query '"프랑켄슈타인" "블루스퀘어"': 2608 total results
Retrieved 100 articles in this request
API Response for query '"프랑켄슈타인" "블루스퀘어"': 2608 total results
Retrieved 100 articles in this request


KeyboardInterrupt: 

기능을 나누어, URL을 긁어온 후, 기사에 들어가서 html 태그 중 본문 부분을 긁어옴.
안긁히는거, 쓸데없는거 긁는거 고쳐줘야 함.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

def get_domain(url):
    return urlparse(url).netloc

def parse_naver(soup):
    article = soup.find('article', id='dic_area')
    if article:
        paragraphs = article.find_all(['p', 'br'])
        return [p.get_text().strip() for p in paragraphs if p.get_text().strip()]
    return []

def parse_others(soup):
    # 다른 도메인을 위한 일반적인 파싱 방법
    paragraphs = soup.find_all(['p', 'br'])
    return [p.get_text().strip() for p in paragraphs if p.get_text().strip()]

def get_article_content(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        domain = get_domain(url)
        if 'naver.com' in domain:
            return parse_naver(soup)
        else:
            return parse_others(soup)
    except Exception as e:
        print(f"Error processing {url}: {str(e)}")
        return []

def main():
    # CSV 파일 읽기
    df = pd.read_csv('/content/drive/MyDrive/crawl/data/musical_articles.csv')

    # 각 URL에 대해 처리
    for index, row in df.iterrows():
        url = row['Article URL']
        print(f"Processing: {url}")

        paragraphs = get_article_content(url)

        # 파싱된 문단을 DataFrame에 추가
        for i, para in enumerate(paragraphs, 1):
            col_name = f'Paragraph {i}'
            if col_name not in df.columns:
                df[col_name] = ''
            df.at[index, col_name] = para

        # 처리 중간중간 저장 (예: 10개 URL마다)
        if (index + 1) % 10 == 0:
            df.to_csv('/content/drive/MyDrive/crawl/data/musical_articles_updated.csv', index=False)
            print(f"Saved progress after processing {index + 1} URLs")

    # 최종 결과 저장
    df.to_csv('/content/drive/MyDrive/crawl/data/musical_articles_updated.csv', index=False)
    print("All data processed and saved.")

if __name__ == "__main__":
    main()

Processing: https://m.entertain.naver.com/article/629/0000305794
Processing: https://n.news.naver.com/mnews/article/081/0003466006?sid=103
Processing: http://www.newsworker.co.kr/news/articleView.html?idxno=340187


KeyboardInterrupt: 